# EA-TTM — Zero-Shot TTM Validation Benchmark

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 03 — Zero-shot validation benchmark  
**Scope:** Establish non-trained forecasting baselines on the **validation interval only** and record the compute profile of zero-shot TTM inference.

This phase performs **no model training** and does **not** evaluate the locked final-test interval.

## Locked upstream contracts

Phase 01 established:

$$
\mathbf{X}
\in
\mathbb{R}^{B\times512\times1}
\quad\longrightarrow\quad
\hat{\mathbf{Y}}_{\mathrm{TTM}}
\in
\mathbb{R}^{B\times28\times1}.
$$

Phase 02 established:

$$
o_{\mathrm{val}}=1885,
$$

with validation context:

$$
d_{1374}:d_{1885}
$$

and validation target:

$$
d_{1886}:d_{1913}.
$$

The locked final test $d_{1914}:d_{1941}$ is not loaded into this benchmark.

## Baseline definitions

### Weekly seasonal naive

Let the final seven observed values before the validation origin be:

$$
\mathbf{s}
=
[y_{o-6},\ldots,y_o].
$$

The 28-step seasonal-naive forecast repeats this seven-day pattern:

$$
\hat y^{SN}_{o+h}
=
y_{o-6+((h-1)\bmod 7)},
\qquad
h=1,\ldots,28.
$$

### Zero-shot TTM

The pre-trained TTM is evaluated without retail-domain parameter updates:

$$
\hat{\mathbf{Y}}^{ZS}
=
f_{\mathrm{TTM}}
\left(
\mathbf{X};
\theta_{\mathrm{pretrained}}
\right).
$$

Predictions are evaluated **without clipping, rounding, or other post-processing**.

## Evaluation metrics

For $N$ forecast observations with actual values $y_i$ and predictions $\hat y_i$:

$$
\mathrm{MSE}
=
\frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat y_i)^2,
$$

$$
\mathrm{RMSE}
=
\sqrt{\mathrm{MSE}},
$$

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
|y_i-\hat y_i|,
$$

and

$$
\mathrm{WAPE}
=
\frac{
\sum_{i=1}^{N}|y_i-\hat y_i|
}{
\sum_{i=1}^{N}|y_i|
}.
$$

In [1]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

## Runtime and project preflight

Each execution-critical notebook validates the persistent artifacts it depends on. Colab runtime state is treated as temporary; Google Drive is the project source of truth.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")
MODEL_SNAPSHOT_DIR = PROJECT_ROOT / "models" / "pretrained_ttm"
PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"
RUN_DIR = PROJECT_ROOT / "runs" / "03_zero_shot_validation"

for directory in (RESULTS_TABLE_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

required_project_artifacts = [
    MODEL_SNAPSHOT_DIR / "config.json",
    METADATA_DIR / "01_TTM_MODEL_CONTRACT.json",
    METADATA_DIR / "02_DATA_MANIFEST.json",
    METADATA_DIR / "02_SPLIT_MANIFEST.json",
    PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
]

missing_project_artifacts = [
    str(path)
    for path in required_project_artifacts
    if not path.exists()
]

assert not missing_project_artifacts, (
    "Required upstream project artifacts are missing: "
    + "; ".join(missing_project_artifacts)
)

Mounted at /content/drive


In [3]:
TSFM_DISTRIBUTION_VERSION = "0.3.9"
TSFM_GIT_REPOSITORY = "https://github.com/ibm-granite/granite-tsfm.git"
TSFM_GIT_TAG = f"v{TSFM_DISTRIBUTION_VERSION}"
TSFM_SOURCE_LOCK = METADATA_DIR / "00_TSFM_SOURCE_LOCK.txt"
RUNTIME_VENDOR_ROOT = Path("/content/ea_ttm_vendor")
TSFM_SOURCE_ROOT = RUNTIME_VENDOR_ROOT / "granite-tsfm"

RUNTIME_VENDOR_ROOT.mkdir(parents=True, exist_ok=True)

def module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def run_command(command: list[str], cwd: Path | None = None) -> str:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()

if not module_available("tsfm_public"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            f"granite-tsfm=={TSFM_DISTRIBUTION_VERSION}",
        ],
        check=True,
    )
    importlib.invalidate_caches()

if not module_available("tsfm_public"):
    if TSFM_SOURCE_LOCK.exists():
        tsfm_source_commit = TSFM_SOURCE_LOCK.read_text(
            encoding="utf-8"
        ).strip()
    else:
        tag_query = subprocess.run(
            [
                "git",
                "ls-remote",
                "--tags",
                TSFM_GIT_REPOSITORY,
                f"refs/tags/{TSFM_GIT_TAG}",
            ],
            check=False,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if tag_query:
            tsfm_source_commit = tag_query.split()[0]
        else:
            head_query = run_command(
                [
                    "git",
                    "ls-remote",
                    TSFM_GIT_REPOSITORY,
                    "refs/heads/main",
                ]
            )
            tsfm_source_commit = head_query.split()[0]

        TSFM_SOURCE_LOCK.write_text(tsfm_source_commit, encoding="utf-8")

    if TSFM_SOURCE_ROOT.exists():
        subprocess.run(["rm", "-rf", str(TSFM_SOURCE_ROOT)], check=True)

    run_command(
        [
            "git",
            "clone",
            "--quiet",
            "--no-checkout",
            TSFM_GIT_REPOSITORY,
            str(TSFM_SOURCE_ROOT),
        ]
    )
    run_command(
        ["git", "checkout", "--quiet", tsfm_source_commit],
        cwd=TSFM_SOURCE_ROOT,
    )

    sys.path.insert(0, str(TSFM_SOURCE_ROOT))
    importlib.invalidate_caches()

assert module_available("tsfm_public"), (
    "IBM TSFM source namespace `tsfm_public` is unavailable."
)

from tsfm_public.models.tinytimemixer.modeling_tinytimemixer import (
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.time_series_preprocessor import (
    DEFAULT_FREQUENCY_MAPPING,
)

## Reproducibility state

The project seed remains fixed at:

$$
s=2026.
$$

No stochastic training occurs in this phase, but the deterministic project state remains active.

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), "CUDA GPU is required for the benchmark."
DEVICE = torch.device("cuda")

## Upstream contract verification

The stored model, dataset, and split manifests are read directly. This notebook does not recreate upstream decisions.

In [5]:
model_contract = json.loads(
    (METADATA_DIR / "01_TTM_MODEL_CONTRACT.json").read_text(encoding="utf-8")
)

data_manifest = json.loads(
    (METADATA_DIR / "02_DATA_MANIFEST.json").read_text(encoding="utf-8")
)

split_manifest = json.loads(
    (METADATA_DIR / "02_SPLIT_MANIFEST.json").read_text(encoding="utf-8")
)

assert model_contract["requested_contract"]["context_length"] == 512
assert model_contract["requested_contract"]["prediction_length"] == 28
assert model_contract["requested_contract"]["frequency"] == "D"
assert model_contract["requested_contract"]["prefer_l1_loss"] is False

assert split_manifest["context_length"] == 512
assert split_manifest["forecast_horizon"] == 28
assert split_manifest["validation_origin"] == 1885
assert split_manifest["locked_test_interval"] == ["d_1914", "d_1941"]

DATASET_SHA256 = data_manifest["dataset_sha256"]
SPLIT_SHA256 = split_manifest["split_sha256"]

CONTEXT_LENGTH = int(split_manifest["context_length"])
FORECAST_HORIZON = int(split_manifest["forecast_horizon"])
VALIDATION_ORIGIN = int(split_manifest["validation_origin"])
FREQUENCY = model_contract["requested_contract"]["frequency"]

frequency_token_value = DEFAULT_FREQUENCY_MAPPING[FREQUENCY]

print("Dataset contract:", DATASET_SHA256)
print("Split contract:", SPLIT_SHA256)
print("Frequency token:", frequency_token_value)

Dataset contract: 83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59
Split contract: 5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447
Frequency token: 8


## Validation tensor construction

Only the validation context and validation target are materialized:

$$
\mathbf{X}_{val}
=
[d_{1374},\ldots,d_{1885}]
$$

and

$$
\mathbf{Y}_{val}
=
[d_{1886},\ldots,d_{1913}].
$$

In [6]:
sales_path = PROCESSED_M5_DIR / "sales_train_evaluation.parquet"

identifier_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

context_start = VALIDATION_ORIGIN - CONTEXT_LENGTH + 1
context_end = VALIDATION_ORIGIN
target_start = VALIDATION_ORIGIN + 1
target_end = VALIDATION_ORIGIN + FORECAST_HORIZON

context_columns = [
    f"d_{day}"
    for day in range(context_start, context_end + 1)
]

target_columns = [
    f"d_{day}"
    for day in range(target_start, target_end + 1)
]

required_columns = identifier_columns + context_columns + target_columns

validation_frame = pd.read_parquet(
    sales_path,
    columns=required_columns,
)

assert validation_frame.shape[0] == 30_490
assert len(context_columns) == 512
assert len(target_columns) == 28

validation_context = validation_frame[
    context_columns
].to_numpy(dtype=np.float32, copy=True)

validation_target = validation_frame[
    target_columns
].to_numpy(dtype=np.float32, copy=True)

assert validation_context.shape == (30_490, 512)
assert validation_target.shape == (30_490, 28)
assert np.isfinite(validation_context).all()
assert np.isfinite(validation_target).all()

print("Validation context shape:", validation_context.shape)
print("Validation target shape:", validation_target.shape)

Validation context shape: (30490, 512)
Validation target shape: (30490, 28)


## Weekly seasonal-naive forecast

The baseline repeats the final observed seven-day demand pattern and therefore uses only information available at the validation origin.

In [7]:
last_observed_week = validation_context[:, -7:]

seasonal_naive_prediction = np.tile(
    last_observed_week,
    (1, math.ceil(FORECAST_HORIZON / 7)),
)[:, :FORECAST_HORIZON]

assert seasonal_naive_prediction.shape == validation_target.shape

## Load the frozen zero-shot TTM

The model is loaded from the exact Phase 01 snapshot. No remote model-selection call is made.

In [8]:
zero_shot_ttm = TinyTimeMixerForPrediction.from_pretrained(
    MODEL_SNAPSHOT_DIR
).to(DEVICE)

zero_shot_ttm.eval()

assert int(zero_shot_ttm.config.context_length) == CONTEXT_LENGTH
assert int(zero_shot_ttm.config.prediction_filter_length) == FORECAST_HORIZON
assert str(zero_shot_ttm.config.loss).lower() == "mse"

for parameter in zero_shot_ttm.parameters():
    parameter.requires_grad_(False)

assert sum(
    parameter.numel()
    for parameter in zero_shot_ttm.parameters()
    if parameter.requires_grad
) == 0

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

## Inference batch-size audit

Inference batch size is a resource parameter rather than an architectural hyperparameter. The notebook tests:

$$
\mathcal{B}
=
\{128,\ 256,\ 512\}
$$

and records the largest successful batch on the active GPU.

In [9]:
BATCH_SIZE_CANDIDATES = [128, 256, 512]

def run_forward_batch(
    model: torch.nn.Module,
    batch_values: torch.Tensor,
) -> torch.Tensor:
    batch_size = batch_values.shape[0]

    frequency_token = torch.full(
        (batch_size,),
        int(frequency_token_value),
        dtype=torch.long,
        device=batch_values.device,
    )

    output = model(
        past_values=batch_values,
        freq_token=frequency_token,
        return_dict=True,
    )

    prediction = output.prediction_outputs

    assert prediction.shape == (
        batch_size,
        FORECAST_HORIZON,
        1,
    )

    return prediction

batch_audit_records: list[dict[str, Any]] = []
selected_batch_size = None

probe_source = validation_context[: max(BATCH_SIZE_CANDIDATES)]

for candidate_batch_size in BATCH_SIZE_CANDIDATES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    probe_batch = torch.from_numpy(
        probe_source[:candidate_batch_size]
    ).unsqueeze(-1).to(DEVICE)

    try:
        with torch.no_grad():
            _ = run_forward_batch(zero_shot_ttm, probe_batch)

        torch.cuda.synchronize()

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

        batch_audit_records.append(
            {
                "batch_size": candidate_batch_size,
                "status": "passed",
                "peak_memory_gb": float(peak_memory_gb),
            }
        )

        selected_batch_size = candidate_batch_size

    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise

        batch_audit_records.append(
            {
                "batch_size": candidate_batch_size,
                "status": "cuda_oom",
                "peak_memory_gb": None,
            }
        )

        torch.cuda.empty_cache()
        break

    finally:
        del probe_batch

assert selected_batch_size is not None, (
    "No candidate inference batch size completed successfully."
)

batch_audit_table = pd.DataFrame(batch_audit_records)
batch_audit_table

,batch_size,status,peak_memory_gb
0,128,passed,0.022875
1,256,passed,0.034090
2,512,passed,0.055775


## Zero-shot TTM inference

The full validation set is processed sequentially. No shuffling, augmentation, clipping, rounding, or target-dependent transformation is applied.

In [10]:
zero_shot_predictions = np.empty(
    (validation_context.shape[0], FORECAST_HORIZON),
    dtype=np.float32,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

inference_start_time = time.perf_counter()

with torch.no_grad():
    for batch_start in range(
        0,
        validation_context.shape[0],
        selected_batch_size,
    ):
        batch_end = min(
            batch_start + selected_batch_size,
            validation_context.shape[0],
        )

        batch_values = torch.from_numpy(
            validation_context[batch_start:batch_end]
        ).unsqueeze(-1).to(DEVICE)

        batch_prediction = run_forward_batch(
            zero_shot_ttm,
            batch_values,
        )

        zero_shot_predictions[
            batch_start:batch_end
        ] = (
            batch_prediction
            .squeeze(-1)
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
        )

        del batch_values, batch_prediction

torch.cuda.synchronize()

inference_runtime_seconds = (
    time.perf_counter() - inference_start_time
)

inference_peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

validation_series_per_second = (
    validation_context.shape[0]
    / inference_runtime_seconds
)

assert zero_shot_predictions.shape == validation_target.shape
assert np.isfinite(zero_shot_predictions).all()

print("Zero-shot runtime (s):", round(inference_runtime_seconds, 3))
print("Peak allocated GPU memory (GB):", round(inference_peak_memory_gb, 4))
print("Series per second:", round(validation_series_per_second, 2))

Zero-shot runtime (s): 1.787
Peak allocated GPU memory (GB): 0.0558
Series per second: 17061.1


## Metric computation

The overall metric table is the primary benchmark result. Horizon-wise metrics are saved as diagnostics.

In [11]:
def forecasting_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    actual_64 = actual.astype(np.float64, copy=False)
    predicted_64 = predicted.astype(np.float64, copy=False)

    error = predicted_64 - actual_64
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    mse = float(squared_error.mean())
    rmse = float(np.sqrt(mse))
    mae = float(absolute_error.mean())

    denominator = float(np.abs(actual_64).sum())
    assert denominator > 0.0, "WAPE denominator is zero."

    wape = float(absolute_error.sum() / denominator)

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "wape": wape,
    }

seasonal_naive_metrics = forecasting_metrics(
    validation_target,
    seasonal_naive_prediction,
)

zero_shot_ttm_metrics = forecasting_metrics(
    validation_target,
    zero_shot_predictions,
)

metrics_table = pd.DataFrame(
    [
        {
            "model": "Weekly Seasonal Naive",
            **seasonal_naive_metrics,
        },
        {
            "model": "Zero-shot TTM",
            **zero_shot_ttm_metrics,
        },
    ]
)

metrics_table

,model,mse,rmse,mae,wape
0,Weekly Seasonal Naive,7.614616,2.759459,1.232355,0.888867
1,Zero-shot TTM,4.557264,2.134775,1.021532,0.736806


In [12]:
horizon_metric_records: list[dict[str, Any]] = []

for horizon_index in range(FORECAST_HORIZON):
    horizon_number = horizon_index + 1

    horizon_metric_records.append(
        {
            "model": "Weekly Seasonal Naive",
            "horizon": horizon_number,
            **forecasting_metrics(
                validation_target[:, horizon_index],
                seasonal_naive_prediction[:, horizon_index],
            ),
        }
    )

    horizon_metric_records.append(
        {
            "model": "Zero-shot TTM",
            "horizon": horizon_number,
            **forecasting_metrics(
                validation_target[:, horizon_index],
                zero_shot_predictions[:, horizon_index],
            ),
        }
    )

horizon_metrics_table = pd.DataFrame(horizon_metric_records)

negative_prediction_count = int(
    np.count_nonzero(zero_shot_predictions < 0.0)
)

negative_prediction_rate = float(
    negative_prediction_count
    / zero_shot_predictions.size
)

mse_improvement_percent = float(
    100.0
    * (
        seasonal_naive_metrics["mse"]
        - zero_shot_ttm_metrics["mse"]
    )
    / seasonal_naive_metrics["mse"]
)

wape_improvement_percent = float(
    100.0
    * (
        seasonal_naive_metrics["wape"]
        - zero_shot_ttm_metrics["wape"]
    )
    / seasonal_naive_metrics["wape"]
)

## Persist benchmark artifacts

Predictions are saved with the exact validation targets so later subgroup analyses do not require repeating zero-shot inference.

In [13]:
METRICS_PATH = RESULTS_TABLE_DIR / "03_VALIDATION_BASELINE_METRICS.csv"
HORIZON_METRICS_PATH = RESULTS_TABLE_DIR / "03_VALIDATION_HORIZON_METRICS.csv"
BATCH_AUDIT_PATH = RESULTS_TABLE_DIR / "03_INFERENCE_BATCH_AUDIT.csv"
PREDICTIONS_PATH = RUN_DIR / "03_VALIDATION_PREDICTIONS.parquet"
RUN_SUMMARY_PATH = METADATA_DIR / "03_RUN_SUMMARY.json"

metrics_table.to_csv(METRICS_PATH, index=False)
horizon_metrics_table.to_csv(HORIZON_METRICS_PATH, index=False)
batch_audit_table.to_csv(BATCH_AUDIT_PATH, index=False)

prediction_frame = validation_frame[identifier_columns].copy()

for horizon_index, target_day in enumerate(target_columns):
    horizon_number = horizon_index + 1

    prediction_frame[
        f"actual_h{horizon_number:02d}_{target_day}"
    ] = validation_target[:, horizon_index]

    prediction_frame[
        f"seasonal_naive_h{horizon_number:02d}"
    ] = seasonal_naive_prediction[:, horizon_index]

    prediction_frame[
        f"zero_shot_ttm_h{horizon_number:02d}"
    ] = zero_shot_predictions[:, horizon_index]

prediction_frame.to_parquet(
    PREDICTIONS_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

assert PREDICTIONS_PATH.exists()

In [14]:
run_summary = {
    "status": "PHASE_03_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "validation_origin": VALIDATION_ORIGIN,
    "validation_target": [target_columns[0], target_columns[-1]],
    "series_count": int(validation_context.shape[0]),
    "selected_batch_size": int(selected_batch_size),
    "inference_runtime_seconds": float(inference_runtime_seconds),
    "inference_peak_memory_gb": float(inference_peak_memory_gb),
    "validation_series_per_second": float(validation_series_per_second),
    "seasonal_naive_metrics": seasonal_naive_metrics,
    "zero_shot_ttm_metrics": zero_shot_ttm_metrics,
    "zero_shot_negative_prediction_count": negative_prediction_count,
    "zero_shot_negative_prediction_rate": negative_prediction_rate,
    "zero_shot_mse_improvement_vs_seasonal_naive_percent": mse_improvement_percent,
    "zero_shot_wape_improvement_vs_seasonal_naive_percent": wape_improvement_percent,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(run_summary, indent=2),
    encoding="utf-8",
)

1084

In [15]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 03 — Zero-shot TTM validation benchmark completed.",
    "",
    "## Locked data/model contracts",
    f"- Dataset SHA-256: `{DATASET_SHA256}`",
    f"- Split SHA-256: `{SPLIT_SHA256}`",
    f"- Context length: {CONTEXT_LENGTH}",
    f"- Forecast horizon: {FORECAST_HORIZON}",
    f"- Validation target: {target_columns[0]} through {target_columns[-1]}",
    "",
    "## Validation benchmark",
    f"- Seasonal-naive MSE: {seasonal_naive_metrics['mse']:.6f}",
    f"- Zero-shot TTM MSE: {zero_shot_ttm_metrics['mse']:.6f}",
    f"- Seasonal-naive WAPE: {seasonal_naive_metrics['wape']:.6f}",
    f"- Zero-shot TTM WAPE: {zero_shot_ttm_metrics['wape']:.6f}",
    "",
    "## Inference resource contract",
    f"- Selected batch size: {selected_batch_size}",
    f"- Peak allocated GPU memory (GB): {inference_peak_memory_gb:.6f}",
    f"- Runtime (s): {inference_runtime_seconds:.3f}",
    "",
    "## Next phase",
    "Head-probe the Slim TTM Decoder and Forecasting Head on the retail training origins",
    "while keeping the pre-trained TTM backbone frozen.",
]

status_path.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

810

## Phase validation

Phase 03 passes only if:

$$
\begin{aligned}
&\text{only the validation target is evaluated}
\\
&\land\ \text{TTM parameters remain frozen}
\\
&\land\ \hat{\mathbf{Y}}^{ZS}
\in
\mathbb{R}^{30490\times28}
\\
&\land\ \text{all predictions are finite}
\\
&\land\ \text{benchmark artifacts are persisted}.
\end{aligned}
$$

In [16]:
phase_validation = {
    "validation_interval_only": (
        target_columns[0] == "d_1886"
        and target_columns[-1] == "d_1913"
    ),
    "ttm_frozen": (
        sum(
            parameter.numel()
            for parameter in zero_shot_ttm.parameters()
            if parameter.requires_grad
        )
        == 0
    ),
    "prediction_shape_valid": (
        zero_shot_predictions.shape == (30_490, 28)
    ),
    "predictions_finite": bool(
        np.isfinite(zero_shot_predictions).all()
    ),
    "metrics_saved": METRICS_PATH.exists(),
    "predictions_saved": PREDICTIONS_PATH.exists(),
    "run_summary_saved": RUN_SUMMARY_PATH.exists(),
}

assert all(phase_validation.values()), (
    f"Phase 03 validation failed: {phase_validation}"
)

phase_validation

{'validation_interval_only': True,
 'ttm_frozen': True,
 'prediction_shape_valid': True,
 'predictions_finite': True,
 'metrics_saved': True,
 'predictions_saved': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [17]:
return_block = {
    "status": "PHASE_03_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "validation_target": [target_columns[0], target_columns[-1]],
    "series_count": int(validation_context.shape[0]),
    "context_length": CONTEXT_LENGTH,
    "forecast_horizon": FORECAST_HORIZON,
    "frequency": FREQUENCY,
    "frequency_token": int(frequency_token_value),
    "selected_batch_size": int(selected_batch_size),
    "inference_runtime_seconds": float(inference_runtime_seconds),
    "inference_peak_memory_gb": float(inference_peak_memory_gb),
    "validation_series_per_second": float(validation_series_per_second),
    "seasonal_naive_metrics": seasonal_naive_metrics,
    "zero_shot_ttm_metrics": zero_shot_ttm_metrics,
    "zero_shot_negative_prediction_count": negative_prediction_count,
    "zero_shot_negative_prediction_rate": negative_prediction_rate,
    "zero_shot_mse_improvement_vs_seasonal_naive_percent": mse_improvement_percent,
    "zero_shot_wape_improvement_vs_seasonal_naive_percent": wape_improvement_percent,
    "artifacts": {
        "metrics": str(METRICS_PATH),
        "horizon_metrics": str(HORIZON_METRICS_PATH),
        "batch_audit": str(BATCH_AUDIT_PATH),
        "validation_predictions": str(PREDICTIONS_PATH),
        "run_summary": str(RUN_SUMMARY_PATH),
        "project_status": str(status_path),
    },
}

print("PHASE 03 RETURN BLOCK")
print(json.dumps(return_block, indent=2))

PHASE 03 RETURN BLOCK
{
  "status": "PHASE_03_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "validation_target": [
    "d_1886",
    "d_1913"
  ],
  "series_count": 30490,
  "context_length": 512,
  "forecast_horizon": 28,
  "frequency": "D",
  "frequency_token": 8,
  "selected_batch_size": 512,
  "inference_runtime_seconds": 1.7871065509999653,
  "inference_peak_memory_gb": 0.0558013916015625,
  "validation_series_per_second": 17061.097998291985,
  "seasonal_naive_metrics": {
    "mse": 7.614616033359884,
    "rmse": 2.7594593733845554,
    "mae": 1.2323548704493277,
    "wape": 0.8888669224907192
  },
  "zero_shot_ttm_metrics": {
    "mse": 4.557263813520326,
    "rmse": 2.1347748859119373,
    "mae": 1.0215323422208904,
    "wape": 0.7368058755052851
  },
  "zero_shot_negative_prediction_count": 5333,
  "zero_shot_negative_prediction_rate": 0.0